### Importation des bibliothèques

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             precision_recall_curve, RocCurveDisplay)
from imblearn.over_sampling import SMOTE

import mlflow
import mlflow.sklearn

import warnings
warnings.filterwarnings('ignore')

### Chargement des données

In [ ]:
# Upload depuis Google Colab
from google.colab import files
uploaded = files.upload()   # sélectionner creditcard.csv

import io
df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))

# ── OU si le fichier est déjà disponible ──
# df = pd.read_csv('data/raw/creditcard.csv')

print(f"Shape : {df.shape}")
df.head()

### Étape 1 — Analyse exploratoire (EDA)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Déséquilibre des classes
print(df['Class'].value_counts())
print(f"\nTaux de fraude : {df['Class'].mean()*100:.3f} %")

df['Class'].value_counts().plot(kind='bar', figsize=(6, 4))
plt.title('Distribution des classes (0=Normal, 1=Fraude)')
plt.xlabel('Classe')
plt.ylabel('Nombre de transactions')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution du montant par classe
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df[df['Class']==0]['Amount'], bins=50, color='steelblue', alpha=0.7)
axes[0].set_title('Montant — Transactions normales')
axes[0].set_xlabel('Montant (€)')

axes[1].hist(df[df['Class']==1]['Amount'], bins=50, color='red', alpha=0.7)
axes[1].set_title('Montant — Transactions frauduleuses')
axes[1].set_xlabel('Montant (€)')

plt.tight_layout()
plt.show()

In [ ]:
# Matrice de corrélation
plt.figure(figsize=(14, 10))
corr = df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', fmt='.1f', annot=False, linewidths=0.3)
plt.title('Matrice de corrélation')
plt.tight_layout()
plt.show()

### Étape 2 — Nettoyage et préparation des données

In [ ]:
# Vérification des valeurs manquantes
print("Valeurs manquantes :")
print(df.isnull().sum().sum(), "au total")

# Vérification du seuil (Data Quality Gate)
missing_rate = df.isnull().mean().max()
if missing_rate > 0.05:
    raise ValueError(f"Pipeline bloqué : taux de valeurs manquantes = {missing_rate:.2%} > 5%")
else:
    print(f"Qualité des données OK (manquants : {missing_rate:.2%})")

In [ ]:
# Normalisation de Time et Amount (les seules features non-PCA)
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled']   = scaler.fit_transform(df[['Time']])

df = df.drop(columns=['Time', 'Amount'])
print("Normalisation effectuée")
df.head()

In [ ]:
# Séparation des features et de la cible
X = df.drop(columns=['Class'])
y = df['Class']

# Split stratifié : 70% train / 15% val / 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3,
                                                     stratify=y, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5,
                                                  stratify=y_temp, random_state=42)

print(f"Train : {X_train.shape[0]} lignes | Val : {X_val.shape[0]} | Test : {X_test.shape[0]}")
print(f"Fraudes train : {y_train.sum()} | val : {y_val.sum()} | test : {y_test.sum()}")

In [ ]:
# Gestion du déséquilibre avec SMOTE (sur le train uniquement)
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

print(f"Après SMOTE — Train : {X_train_res.shape[0]} lignes")
print(f"Fraudes : {y_train_res.sum()} | Normal : {(y_train_res==0).sum()}")

### Étape 3 — Entraînement et comparaison des modèles

In [ ]:
mlflow.set_experiment("fraud-detection-mlops")

def evaluate(model, X_v, y_v, model_name):
    y_proba = model.predict_proba(X_v)[:, 1]
    y_pred  = model.predict(X_v)
    auc_roc = roc_auc_score(y_v, y_proba)
    pr_auc  = average_precision_score(y_v, y_proba)
    print(f"\n--- {model_name} ---")
    print(f"ROC-AUC : {auc_roc:.4f} | PR-AUC : {pr_auc:.4f}")
    print(classification_report(y_v, y_pred, target_names=['Normal', 'Fraude']))
    return auc_roc, pr_auc, y_proba

In [ ]:
# Modèle 1 : Régression Logistique
with mlflow.start_run(run_name="LogisticRegression"):
    lr = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
    lr.fit(X_train_res, y_train_res)

    auc, prauc, _ = evaluate(lr, X_val, y_val, "Régression Logistique")

    mlflow.log_param("model",        "LogisticRegression")
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("smote",        True)
    mlflow.log_metric("roc_auc",     auc)
    mlflow.log_metric("pr_auc",      prauc)
    mlflow.sklearn.log_model(lr, "model")

In [ ]:
# Modèle 2 : Random Forest
with mlflow.start_run(run_name="RandomForest"):
    rf = RandomForestClassifier(n_estimators=100, class_weight='balanced',
                                random_state=42, n_jobs=-1)
    rf.fit(X_train_res, y_train_res)

    auc, prauc, _ = evaluate(rf, X_val, y_val, "Random Forest")

    mlflow.log_param("model",         "RandomForest")
    mlflow.log_param("n_estimators",  100)
    mlflow.log_param("class_weight",  "balanced")
    mlflow.log_param("smote",         True)
    mlflow.log_metric("roc_auc",      auc)
    mlflow.log_metric("pr_auc",       prauc)
    mlflow.sklearn.log_model(rf, "model")

In [ ]:
# Modèle 3 : Isolation Forest (non supervisé)
with mlflow.start_run(run_name="IsolationForest"):
    iso = IsolationForest(contamination=0.002, random_state=42, n_jobs=-1)
    iso.fit(X_train)

    preds_iso = (iso.predict(X_val) == -1).astype(int)
    auc_iso   = roc_auc_score(y_val, preds_iso)
    prauc_iso = average_precision_score(y_val, preds_iso)

    print("\n--- Isolation Forest ---")
    print(f"ROC-AUC : {auc_iso:.4f} | PR-AUC : {prauc_iso:.4f}")
    print(classification_report(y_val, preds_iso, target_names=['Normal', 'Fraude']))

    mlflow.log_param("model",         "IsolationForest")
    mlflow.log_param("contamination", 0.002)
    mlflow.log_metric("roc_auc",      auc_iso)
    mlflow.log_metric("pr_auc",       prauc_iso)

### Étape 4 — Évaluation finale du meilleur modèle

In [ ]:
# Meilleur modèle : Random Forest → évaluation sur le test set
auc_test, prauc_test, y_proba_test = evaluate(rf, X_test, y_test, "Random Forest — TEST")

# Matrice de confusion
cm = confusion_matrix(y_test, rf.predict(X_test))
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Prédit Normal', 'Prédit Fraude'],
            yticklabels=['Réel Normal', 'Réel Fraude'])
plt.title('Matrice de confusion — Random Forest')
plt.tight_layout()
plt.show()

In [ ]:
# Courbe Précision-Rappel
precision, recall, thresholds = precision_recall_curve(y_test, y_proba_test)

plt.figure(figsize=(8, 5))
plt.plot(recall, precision, color='steelblue', lw=2)
plt.xlabel('Rappel')
plt.ylabel('Précision')
plt.title(f'Courbe Précision-Rappel (PR-AUC = {prauc_test:.4f})')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Importance des variables
importance = pd.Series(rf.feature_importances_, index=X.columns)
importance.sort_values().tail(15).plot(kind='barh', figsize=(10, 6))
plt.title('Top 15 variables importantes — Random Forest')
plt.tight_layout()
plt.show()

### Étape 5 — Détection de dérive des données

In [ ]:
# Simulation de data drift (après t0)
df_drift = X_test.copy()
n_drift   = int(len(df_drift) * 0.20)
idx_drift = np.random.choice(df_drift.index, n_drift, replace=False)

# Amount × 1.3 sur 20% des transactions
df_drift.loc[idx_drift, 'Amount_scaled'] *= 1.3

# Missing rate : 0% → 5% sur V1
idx_missing = np.random.choice(df_drift.index, int(len(df_drift)*0.05), replace=False)
df_drift.loc[idx_missing, 'V1'] = np.nan
df_drift = df_drift.fillna(df_drift.median())

# Comparaison des distributions avant/après drift
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(X_test['Amount_scaled'], bins=50, alpha=0.6, label='Original', color='steelblue')
axes[0].hist(df_drift['Amount_scaled'], bins=50, alpha=0.6, label='Drift', color='red')
axes[0].set_title('Distribution Amount — Avant vs Après drift')
axes[0].legend()

# Performance après drift
y_pred_drift = rf.predict(df_drift)
auc_drift    = roc_auc_score(y_test, rf.predict_proba(df_drift)[:, 1])
axes[1].bar(['Avant drift', 'Après drift'], [auc_test, auc_drift], color=['steelblue','red'])
axes[1].set_title('ROC-AUC avant et après data drift')
axes[1].set_ylim(0.8, 1.0)

plt.tight_layout()
plt.show()

print(f"ROC-AUC avant drift : {auc_test:.4f}")
print(f"ROC-AUC après drift : {auc_drift:.4f}")
if auc_test - auc_drift > 0.01:
    print("ALERTE : dérive détectée — dégradation > 1%")
else:
    print("Pas de dérive significative détectée")

### Conclusion

In [ ]:
print("=" * 60)
print("  RÉSULTATS : FRAUDE BANCAIRE PAR CALIXTE NGUEMO ET CORELIA WILDERVIA")
print("=" * 60)
print(f"  Dataset        : Credit Card Fraud Detection")
print(f"  Transactions   : 284 807 | Fraudes : 492 (0.17%)")
print()
print(f"  Modèles testés :")
print(f"  - Régression Logistique")
print(f"  - Random Forest      ← Champion")
print(f"  - Isolation Forest")
print()
print(f" Modèle Champion : Random Forest")
print(f"  ROC-AUC (test)  : {auc_test:.4f}")
print(f"  PR-AUC  (test)  : {prauc_test:.4f}")
print()
print("  RECOMMANDATIONS :")
print("  - Déployer Random Forest via FastAPI + Docker")
print("  - Surveiller le data drift chaque semaine")
print("  - Recalibrer le seuil de décision selon PR-AUC")
print("  - Réentraîner si ROC-AUC chute de plus de 1%")
print("=" * 60)